In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import gamma
import math
import time

# Dados de treino originais (Naturais de 1 a 10)
X_train = np.arange(1, 11, dtype=float)
y_factorial = np.array([math.factorial(int(n)) for n in X_train], dtype=float)

# Transformação de escala de Chang/Wiktorowicz para o espaço intermediário
y_train_transformed = np.zeros_like(X_train)
for i, n in enumerate(X_train):
    fact = math.factorial(int(n))
    y_train_transformed[i] = 1.0 if fact == 1 else 1.0 / np.log(fact)

print("X de Treino:", X_train)
print("Y Transformado:", y_train_transformed)

In [ ]:
def tsk_inference(X, centers, sigmas, y_target=None, ridge_lambda=1.0):
    X = np.atleast_1d(X)
    N = len(X)
    R = len(centers)
    
    # 1. Calcular ativação Gaussiana normal
    W = np.zeros((N, R))
    for i in range(N):
        for j in range(R):
            W[i, j] = np.exp(-((X[i] - centers[j])**2) / (2 * (sigmas[j]**2) + 1e-5))
            
    # CORREÇÃO CRUCIAL PARA EXTRAPOLAÇÃO (Wiktorowicz, 2021):
    # Se o ponto X estiver fora das fronteiras dos centros de treino, 
    # forçamos a ativação total da regra da ponta correspondente.
    for i in range(N):
        if X[i] < centers.min():
            # Se está nos negativos, a regra do extremo esquerdo assume 100%
            idx_min = np.argmin(centers)
            W[i, :] = 0.0
            W[i, idx_min] = 1.0
        elif X[i] > centers.max():
            # Se extrapolar para além de 10, a regra do extremo direito assume 100%
            idx_max = np.argmax(centers)
            W[i, :] = 0.0
            W[i, idx_max] = 1.0
            
    # Normalização dos pesos de inferência
    row_sums = W.sum(axis=1, keepdims=True)
    W_norm = np.where(row_sums > 1e-12, W / row_sums, 1.0 / R)
    
    # 2. Matriz de Design Global
    X_hat = np.zeros((N, 2 * R))
    for j in range(R):
        X_hat[:, 2*j] = W_norm[:, j] * X
        X_hat[:, 2*j + 1] = W_norm[:, j]
        
    # 3. Resolução via Ridge Regression
    if y_target is not None:
        A = X_hat.T @ X_hat + ridge_lambda * np.eye(2 * R)
        try:
            P = np.linalg.inv(A) @ X_hat.T @ y_target
        except np.linalg.LinAlgError:
            P = np.linalg.pinv(A) @ X_hat.T @ y_target
        return X_hat @ P, P
    else:
        return X_hat

In [ ]:
def particle_swarm_optimization(X, y_trans, num_rules=4, pop_size=40, iterations=40, seed=42):
    np.random.seed(seed)
    # Centros concisos focados na região positiva (Chang, 2025)
    centers = np.linspace(X.min(), X.max(), num_rules)
    
    w, c1, c2 = 0.6, 1.8, 1.8
    # Inicialização variada para dar dinâmica real de busca ao enxame
    position = np.random.uniform(1.0, 3.5, size=(pop_size, num_rules))
    velocity = np.random.uniform(-0.1, 0.1, size=(pop_size, num_rules))
    
    pbest_position = position.copy()
    pbest_fitness = np.array([np.mean((y_trans - tsk_inference(X, centers, ind, y_trans, ridge_lambda=1e-2)[0])**2) for ind in position])
    gbest_idx = np.argmin(pbest_fitness)
    gbest_position = pbest_position[gbest_idx].copy()
    
    history = []
    for it in range(iterations):
        for i in range(pop_size):
            r1, r2 = np.random.rand(num_rules), np.random.rand(num_rules)
            velocity[i] = w * velocity[i] + c1 * r1 * (pbest_position[i] - position[i]) + c2 * r2 * (gbest_position - position[i])
            # Limite elástico menor para permitir o PSO trabalhar sem estagnar de cara
            position[i] = np.clip(position[i] + velocity[i], 0.5, 5.0) 
            
            y_pred, _ = tsk_inference(X, centers, position[i], y_trans, ridge_lambda=1e-2)
            current_mse = np.mean((y_trans - y_pred)**2)
            
            if current_mse < pbest_fitness[i]:
                pbest_fitness[i] = current_mse
                pbest_position[i] = position[i].copy()
                if current_mse < pbest_fitness[gbest_idx]:
                    gbest_position = position[i].copy()
                    gbest_idx = i
        history.append(pbest_fitness[gbest_idx])
    return gbest_position, history, centers

def genetic_algorithm(X, y_trans, num_rules=4, pop_size=40, generations=40, mutation_rate=0.2, seed=42):
    np.random.seed(seed)
    centers = np.linspace(X.min(), X.max(), num_rules)
    population = np.random.uniform(1.0, 3.5, size=(pop_size, num_rules))
    history = []
    
    for gen in range(generations):
        fitness = np.array([np.mean((y_trans - tsk_inference(X, centers, ind, y_trans, ridge_lambda=1e-2)[0])**2) for ind in population])
        best_idx = np.argmin(fitness)
        history.append(fitness[best_idx])
        
        new_population = []
        for _ in range(pop_size):
            candidates = np.random.choice(pop_size, size=3, replace=False)
            winner = candidates[np.argmin(fitness[candidates])]
            new_population.append(population[winner].copy())
            
        new_population = np.array(new_population)
        for i in range(0, pop_size, 2):
            if i+1 < pop_size and np.random.rand() < 0.8:
                alpha = np.random.rand()
                child1 = alpha * new_population[i] + (1 - alpha) * new_population[i+1]
                child2 = alpha * new_population[i+1] + (1 - alpha) * new_population[i]
                new_population[i] = np.clip(child1, 0.5, 5.0)
                new_population[i+1] = np.clip(child2, 0.5, 5.0)
                
        for i in range(pop_size):
            if np.random.rand() < mutation_rate:
                mutation_vector = np.random.normal(0, 0.2, size=num_rules)
                new_population[i] = np.clip(new_population[i] + mutation_vector, 0.5, 5.0)
                
        new_population[0] = population[best_idx]
        population = new_population
        
    final_fitness = np.array([np.mean((y_trans - tsk_inference(X, centers, ind, y_trans, ridge_lambda=1e-2)[0])**2) for ind in population])
    return population[np.argmin(final_fitness)], history, centers

In [ ]:
seeds = [10, 42, 100, 2026, 999]
NUM_REGRAS = 4  # Retornando ao modelo conciso ideal (Chang, 2025)
ITERACOES = 40
LAMBDA_ESTAVEL = 1e-2

pso_mses, pso_rmses, pso_mapes, pso_times, pso_curves = [], [], [], [], []
ga_mses, ga_rmses, ga_mapes, ga_times, ga_curves = [], [], [], [], []

def calculate_advanced_metrics(y_real, y_pred_transformed):
    y_pred_real = np.exp(1.0 / y_pred_transformed)
    mse = np.mean((y_real - y_pred_real)**2)
    rmse = np.sqrt(mse)
    mape = np.mean(np.abs((y_real - y_pred_real) / y_real)) * 100
    return mse, rmse, mape

# --- EXECUÇÃO PSO ---
for s in seeds:
    t0 = time.time()
    best_sigmas, hist, centers = particle_swarm_optimization(X_train, y_train_transformed, num_rules=NUM_REGRAS, iterations=ITERACOES, seed=s)
    pso_times.append(time.time() - t0)
    pso_curves.append(hist)
    
    y_pred_trans, _ = tsk_inference(X_train, centers, best_sigmas, y_train_transformed, ridge_lambda=LAMBDA_ESTAVEL)
    _, rmse, mape = calculate_advanced_metrics(y_factorial, y_pred_trans)
    pso_mses.append(hist[-1])
    pso_rmses.append(rmse)
    pso_mapes.append(mape)

# --- EXECUÇÃO GA ---
for s in seeds:
    t0 = time.time()
    best_sigmas, hist, centers = genetic_algorithm(X_train, y_train_transformed, num_rules=NUM_REGRAS, generations=ITERACOES, seed=s)
    ga_times.append(time.time() - t0)
    ga_curves.append(hist)
    
    y_pred_trans, _ = tsk_inference(X_train, centers, best_sigmas, y_train_transformed, ridge_lambda=LAMBDA_ESTAVEL)
    _, rmse, mape = calculate_advanced_metrics(y_factorial, y_pred_trans)
    ga_mses.append(hist[-1])
    ga_rmses.append(rmse)
    ga_mapes.append(mape)

print("\n" + "="*60)
print("     TABELA COMPARATIVA ESTATÍSTICA (4 REGRAS POSITIVAS)")
print("="*60)
print(f"Métrica               | PSO              | GA")
print(f"------------------------------------------------------------")
print(f"Melhor MSE (Treino)   | {np.min(pso_mses):.6f}         | {np.min(ga_mses):.6f}")
print(f"Desvio Padrão MSE     | {np.std(pso_mses):.6f}         | {np.std(ga_mses):.6f}")
print(f"Média RMSE (Real)     | {np.mean(pso_rmses):.2f}       | {np.mean(ga_rmses):.2f}")
print(f"Média MAPE (Real %)   | {np.mean(pso_mapes):.2f}%          | {np.mean(ga_mapes):.2f}%")
print(f"Tempo Médio (s)       | {np.mean(pso_times):.4f}           | {np.mean(ga_times):.4f}")
print("="*60)

In [ ]:
# 1. Recuperação dos melhores modelos de cada categoria
best_pso_idx = np.argmin(pso_mses)
best_pso_sigmas, _, centers_pso = particle_swarm_optimization(X_train, y_train_transformed, num_rules=NUM_REGRAS, iterations=ITERACOES, seed=seeds[best_pso_idx])
_, optimal_P_pso = tsk_inference(X_train, centers_pso, best_pso_sigmas, y_train_transformed, ridge_lambda=LAMBDA_ESTAVEL)

best_ga_idx = np.argmin(ga_mses)
best_ga_sigmas, _, ga_centers = genetic_algorithm(X_train, y_train_transformed, num_rules=NUM_REGRAS, generations=ITERACOES, seed=seeds[best_ga_idx])
_, optimal_P_ga = tsk_inference(X_train, ga_centers, best_ga_sigmas, y_train_transformed, ridge_lambda=LAMBDA_ESTAVEL)

# Domínio contínuo focado EXCLUSIVAMENTE nos positivos reais fracionários [1.0, 10.0]
X_pos_continuous = np.linspace(1.0, 10.0, 300)

# Inferências no espaço intermediário de regressão
y_pso_pos_trans = tsk_inference(X_pos_continuous, centers_pso, best_pso_sigmas, ridge_lambda=LAMBDA_ESTAVEL) @ optimal_P_pso
y_ga_pos_trans = tsk_inference(X_pos_continuous, ga_centers, best_ga_sigmas, ridge_lambda=LAMBDA_ESTAVEL) @ optimal_P_ga

# Inversão exponencial estável para reconstrução contínua
pso_factorial_pos = np.exp(1.0 / y_pso_pos_trans)
ga_factorial_pos = np.exp(1.0 / y_ga_pos_trans)

# 2. Renderização da nova figura com 3 gráficos lado a lado (Sem os negativos poluídos)
fig, axs = plt.subplots(1, 3, figsize=(18, 5))

# --- GRÁFICO 1: VELOCIDADE DE CONVERGÊNCIA REAL ---
axs[0].plot(np.mean(pso_curves, axis=0), 'r-', linewidth=2, label='PSO (Média)')
axs[0].plot(np.mean(ga_curves, axis=0), 'b-', linewidth=2, label='GA (Média)')
axs[0].set_title("Curva de Convergência Real (4 Regras)")
axs[0].set_xlabel("Iterações / Gerações")
axs[0].set_ylabel("MSE Médio (Espaço Transformado)")
axs[0].set_yscale('log')
axs[0].grid(True)
axs[0].legend()

# --- GRÁFICO 2: SUPERFÍCIE NO ESPAÇO TRANSFORMADO INTERMEDIÁRIO ---
y_gamma_trans_pos = 1.0 / np.log(gamma(X_pos_continuous + 1))
axs[1].plot(X_pos_continuous, y_gamma_trans_pos, 'g-', label='Meta Analítica $1/\\ln(x!)$', alpha=0.5)
axs[1].plot(X_pos_continuous, y_pso_pos_trans, 'r--', label='Inferência PSO')
axs[1].plot(X_pos_continuous, y_ga_pos_trans, 'b:', label='Inferência GA', linewidth=2)
axs[1].scatter(X_train, y_train_transformed, color='black', zorder=5, label='Pontos de Treino')
axs[1].set_title("Superfície Fuzzy no Espaço Transformado")
axs[1].set_xlabel("Entrada ($x$)")
axs[1].set_ylabel("Valor")
axs[1].grid(True)
axs[1].legend()

# --- GRÁFICO 3: GENERALIZAÇÃO FRACIONÁRIA POSITIVA VS BASELINE GAMMA ---
axs[2].plot(X_pos_continuous, gamma(X_pos_continuous + 1), 'g-', label='Função Gamma $\\Gamma(x+1)$', alpha=0.6)
axs[2].plot(X_pos_continuous, pso_factorial_pos, 'r--', label='Interpolação PSO')
axs[2].plot(X_pos_continuous, ga_factorial_pos, 'b:', label='Interpolação GA', linewidth=2)
axs[2].scatter(X_train, y_factorial, color='black', zorder=5, label='Fatoriais de Treino ($n!$)')
axs[2].set_title("Aproximação Contínua Fracionária")
axs[2].set_xlabel("Entrada Contínua ($x$)")
axs[2].set_ylabel("Valor (Escala Logarítmica)")
axs[2].set_yscale('log')
axs[2].grid(True)
axs[2].legend()

plt.tight_layout()

# Salvamento seguro para enviar ao Overleaf
plt.savefig('resultado_experimento_positivo.png', dpi=300, bbox_inches='tight')
plt.show()